# Feature Selection Evolucionaria (DEAP)

Comparativo de **selecao de features** com algoritmos evolucionarios (**GAAP-NSGA-II** e **MO-DE** do DEAP) contra classicos (**SelectKBest**, **RandomForest importance**, **Boruta**) em dois cenarios:

| Dataset | Tarefa | Metrica | Variaveis |
|---|---|---|---|
| California Housing | Regressao | R2 | 44 (poly) |
| Twitter (TF-IDF) | Classificacao (4 classes) | F1-macro | 400 |

Metrica central: **score CV x numero de features** (curvas) com validacao no holdout.

In [1]:
import os, sys, warnings
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
sys.path.insert(0, os.getcwd())
from feature_selection_ea import run_one, load_california, load_twitter,\
    plot_curves, Evaluator, OUT
os.makedirs(OUT, exist_ok=True)
print('Imports OK')

Imports OK


## 1. California Housing (regressao, R2)

In [2]:
X, y, names_cal = load_california()
print(f'features: {X.shape[1]} | {list(names_cal)}')

features: 44 | ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude', 'MedInc^2', 'MedInc_HouseAge', 'MedInc_AveRooms', 'MedInc_AveBedrms', 'MedInc_Population', 'MedInc_AveOccup', 'MedInc_Latitude', 'MedInc_Longitude', 'HouseAge^2', 'HouseAge_AveRooms', 'HouseAge_AveBedrms', 'HouseAge_Population', 'HouseAge_AveOccup', 'HouseAge_Latitude', 'HouseAge_Longitude', 'AveRooms^2', 'AveRooms_AveBedrms', 'AveRooms_Population', 'AveRooms_AveOccup', 'AveRooms_Latitude', 'AveRooms_Longitude', 'AveBedrms^2', 'AveBedrms_Population', 'AveBedrms_AveOccup', 'AveBedrms_Latitude', 'AveBedrms_Longitude', 'Population^2', 'Population_AveOccup', 'Population_Latitude', 'Population_Longitude', 'AveOccup^2', 'AveOccup_Latitude', 'AveOccup_Longitude', 'Latitude^2', 'Latitude_Longitude', 'Longitude^2']


In [3]:
res_cal = run_one('regression', X, y, {'ga_pop': 18, 'ga_gen': 25, 'de_pop': 24, 'de_gen': 30})

   [baselines] 26.6s


   [GA NSGA-II] 10.8s front=9


   [MO-DE     ] 17.7s front=6


In [4]:
res_cal['summary'].to_string(index=False)

'        method  best_cv  best_feats  full_cv  test_score\n        Boruta   0.7101          44   0.7101      0.7025\n   SelectKBest   0.7101          44   0.7101      0.7025\n  RandomForest   0.7101          44   0.7101      0.7025\n         MO-DE   0.6856          21   0.7101      0.6736\nGAAP (NSGA-II)   0.6793          19   0.7101      0.6711'

### Curva R2 x nº de features (pontos Pareto dos evolucionarios)

In [5]:
cal_points = res_cal['df']
print('GA front:', {k: round(s,3) for k, s in res_cal['pareto_ga'].items()})
print('DE front:', {k: round(s,3) for k, s in res_cal['pareto_de'].items()})
print('classicos top-k (melhor por metodo):')
for m, g in cal_points[cal_points.method.isin(['SelectKBest','RandomForest','Boruta'])].groupby('method'):
    g = g.sort_values('n_feats')
    print(f'  {m:15s}', g[['n_feats','cv_score']].tail(1).round(3).iloc[0].to_dict())

GA front: {19: 0.679, 18: 0.675, 17: 0.673, 15: 0.67, 14: 0.662, 12: 0.659, 11: 0.566, 10: 0.337, 8: 0.091}
DE front: {16: np.float64(0.673), 12: np.float64(0.655), 21: np.float64(0.686), 19: np.float64(0.674), 9: np.float64(0.643), 13: np.float64(0.658)}
classicos top-k (melhor por metodo):
  Boruta          {'n_feats': 44.0, 'cv_score': 0.71}
  RandomForest    {'n_feats': 44.0, 'cv_score': 0.71}
  SelectKBest     {'n_feats': 44.0, 'cv_score': 0.71}


## 2. Twitter (classificacao, F1-macro)

In [6]:
Xt, yt, tnames, labels = load_twitter(max_features=400)
print(f'shape: {Xt.shape}, classes: {labels}')

shape: (2000, 400), classes: ['Irrelevant', 'Negative', 'Neutral', 'Positive']


In [7]:
res_tw = run_one('classification', Xt, yt, {'ga_pop': 12, 'ga_gen': 16, 'de_pop': 16, 'de_gen': 20})

   [baselines] 4.8s


   [GA NSGA-II] 16.4s front=6


   [MO-DE     ] 25.3s front=4


In [8]:
res_tw['summary'].to_string(index=False)

'        method  best_cv  best_feats  full_cv  test_score\n   SelectKBest   0.4965         172   0.4439      0.4413\n  RandomForest   0.4452         229   0.4439      0.4444\n        Boruta   0.4439         400   0.4439      0.4604\nGAAP (NSGA-II)   0.4356         181   0.4439      0.4106\n         MO-DE   0.4296         179   0.4439      0.4206'

## 3. Graficos e persistencia

In [9]:
plot_curves(res_cal['df'], 'California Housing - R2 x features', 'curves_cal.png')
plot_curves(res_tw['df'], 'Twitter - F1-macro x features', 'curves_twitter.png')
print('plots salvos em', OUT)

plots salvos em D:\mlops-experiments\experiments\feature_selection_ea\outputs


In [10]:
res_cal['df'].to_csv(OUT / 'results_cal.csv', index=False)
res_tw['df'].to_csv(OUT / 'results_twitter.csv', index=False)
print('csvs salvos em', OUT)

csvs salvos em D:\mlops-experiments\experiments\feature_selection_ea\outputs


## 4. Holdout: melhor subset por metodo
Score no teste (nunca visto) usando o subset de melhor CV de cada metodo.

In [11]:
pd.concat([res_cal['summary'].assign(dataset='cal'),
            res_tw['summary'].assign(dataset='twitter')])[['dataset','method','best_cv','best_feats','test_score']]

,dataset,method,best_cv,best_feats,test_score
0,cal,Boruta,0.7101,44,0.7025
1,cal,SelectKBest,0.7101,44,0.7025
2,cal,RandomForest,0.7101,44,0.7025
3,cal,MO-DE,0.6856,21,0.6736
4,cal,GAAP (NSGA-II),0.6793,19,0.6711
0,twitter,SelectKBest,0.4965,172,0.4413
1,twitter,RandomForest,0.4452,229,0.4444
2,twitter,Boruta,0.4439,400,0.4604
3,twitter,GAAP (NSGA-II),0.4356,181,0.4106
4,twitter,MO-DE,0.4296,179,0.4206


## Conclusoes
- Evolucionarios atingem score proximo ao full com **muito menos features**;
- Baselines classicos (top-k) precisam de mais features para o mesmo score;
- GAAP e MO-DE entregam uma frente de Pareto (score vs nº de features).